In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Saving the Goat

An agricultural economist has to sell a *wolf*, a *goat*, and a *cabbage* on a market place.  In order to
reach the market place, she has to cross a river.  The boat that she can use is so small that it can
only accommodate either the goat, the wolf, or the cabbage in addition to the agricultural economist.
Now if the agricultural economist leaves the wolf alone with the goat, the wolf will eat the goat.
If, instead, the agricultural economist leaves the goat with the cabbage, the goat will eat the cabbage.
Is it possible for the agricultural economist to develop a schedule that allows her to cross the river
without either the goat or the cabbage being eaten?

We will encode this problem as a *symbolic transition system* and then solve it with the help of our *constraint solver*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{F}i$ for $i\in\{0,\cdots,n\}$ is the number of farmers on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{W}i$ for $i\in\{0,\cdots,n\}$ is the number of wolves on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{G}i$ for $i\in\{0,\cdots,n\}$ is the number of goats on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of cabbages on the western shore after the 
  $i^{\textrm{th}}$ crossing.

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the function `constr` that creates constraints, and the function `solve`.

In [ ]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

## Auxiliary Functions

The function `start` takes four natural numbers as input:
* `F` is the number of farmers on the western shore,
* `W` is the number of wolves on the western shore,
* `G` is the number of goats on the western shore, and
* `C` is the number of cabbages on the western shore.

It returns `true` in the initial state where everybody is on the western shore.

In [ ]:
let start (F: int) (W: int) (G: int) (C: int) : bool =
    F = 1 && W = 1 && G = 1 && C = 1

The function `goal` takes four natural numbers as input:
* `F` is the number of farmers on the western shore,
* `W` is the number of wolves on the western shore,
* `G` is the number of goats on the western shore, and
* `C` is the number of cabbages on the western shore.

It returns `true` in the state where everybody is on the eastern shore and hence nobody is on the western shore.

In [ ]:
let goal (F: int) (W: int) (G: int) (C: int) : bool =
    F = 0 && W = 0 && G = 0 && C = 0

The function `invariant` takes four natural numbers as input:
* `F` is the number of farmers on the western shore,
* `W` is the number of wolves on the western shore,
* `G` is the number of goats on the western shore, and
* `C` is the number of cabbages on the western shore.

It returns `true` if there is no problem on either shore of the river.  There is no problem if on the shore where there is no farmer, the following conditions hold:
* If the wolf is on that shore, the goat must be on the opposite shore.
* If the goat is on that shore, the cabbage has to be on the opposite shore.

In [ ]:
let invariant (F: int) (W: int) (G: int) (C: int) : bool =
    F = G || (W = F && F = C)

The function `transition` takes 8 arguments:
* `Fα` is the number of farmers on the western shore before the crossing.
* `Wα` is the number of wolves on the western shore before the crossing.
* `Gα` is the number of goats on the western shore before the crossing. 
* `Cα` is the number of cabbages on the western shore before the crossing. 
* `Fβ` is the number of farmers on the western shore after the crossing.
* `Wβ` is the number of wolves on the western shore after the crossing.
* `Gβ` is the number of goats on the western shore after the crossing.
* `Cβ` is the number of cabbages on the western shore after the crossing.

It returns `true` if the crossing is admissible, i.e.:
* The farmer travels in every crossing.
* She takes at most one of her goods with her in the boat.
* If the farmer travels from the western shore to the eastern shore,
  the number of goods on the western shore stays the same or decreases, otherwise it stays the same or increases.

In [ ]:
let transition (Fα: int) (Wα: int) (Gα: int) (Cα: int) (Fβ: int) (Wβ: int) (Gβ: int) (Cβ: int) : bool =
    Fβ = 1 - Fα &&
    ((Fα = 1 && (Wα - Wβ) + (Gα - Gβ) + (Cα - Cβ) <= 1 && Wβ <= Wα && Gβ <= Gα && Cβ <= Cα) ||
     (Fα = 0 && (Wβ - Wα) + (Gβ - Gα) + (Cβ - Cα) <= 1 && Wβ >= Wα && Gβ >= Gα && Cβ >= Cα))

The function `wgcCSP` takes one argument:
- `n` is the number of crossings.

It creates a CSP that tries to solve the problem with `n` crossings.  The variables are ordered by the number of the crossing.  The constraints call the functions defined above.  For example, the constraint `start(F0, W0, G0, C0)` could be created by the expression
```
constr "start(F0, W0, G0, C0)" ["F0"; "W0"; "G0"; "C0"] (fun A -> start A["F0"] A["W0"] A["G0"] A["C0"])
```
In order to avoid writing the variable names twice, we use the following auxiliary functions:
- `vars i` returns the list of the variables `Fi`, `Wi`, `Gi`, and `Ci` that describe the state after the $i^{\textrm{th}}$ crossing.
- `apply4 f A i` applies the function `f` to the values of the variables `Fi`, `Wi`, `Gi`, and `Ci` in the assignment `A`.  As F# functions are *curried*, `apply4 transition A i` is a function that still expects the four values describing the state after the next crossing.  Therefore, `apply4 (apply4 transition A i) A (i+1)` applies `transition` to all eight values.
- `text name xs` returns a string of the form `name(x1, ..., xn)` that is used as the text of the constraint.

In [ ]:
let wgcCSP (n: int) : CSP<int> =
    let vars i = [ sprintf "F%d" i; sprintf "W%d" i; sprintf "G%d" i; sprintf "C%d" i ]
    let apply4 (f: int -> int -> int -> int -> 'R) (A: Map<string, int>) (i: int) : 'R =
        f A[sprintf "F%d" i] A[sprintf "W%d" i] A[sprintf "G%d" i] A[sprintf "C%d" i]
    let text name xs = sprintf "%s(%s)" name (String.concat ", " xs)
    let Start       = constr (text "start" (vars 0)) (vars 0) (fun A -> apply4 start A 0)
    let Goal        = constr (text "goal"  (vars n)) (vars n) (fun A -> apply4 goal  A n)
    let Invariants  = [ for i in 0 .. n - 1 ->
                          constr (text "invariant" (vars i)) (vars i) (fun A -> apply4 invariant A i) ]
    let Transitions = [ for i in 0 .. n - 1 ->
                          constr (text "transition" (vars i @ vars (i+1))) (vars i @ vars (i+1))
                                 (fun A -> apply4 (apply4 transition A i) A (i+1)) ]
    { Variables   = List.concat [ for i in 0 .. n -> vars i ]
      Values      = [ 0; 1 ]
      Constraints = Start :: Goal :: Invariants @ Transitions }

In [ ]:
wgcCSP 3

The function `findSolution` takes no arguments.  It computes a solution to the *wolf-goat-cabbage* problem by trying $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings `n` together with the solution.

In [ ]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match solve (wgcCSP n) with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

Solving the problem should take less than a second.

In [ ]:
#!time
let n, Solution = findSolution ()

In [ ]:
n, Solution

The function `showSolution` takes two arguments:
- `Solution` is a solution of the CSP,
- `n` is the number of crossings.

It prints the solution.  For every $i \in \{0, \cdots, n\}$, it prints the situation after the $i^{\textrm{th}}$ crossing.  The western shore is printed on the left, the eastern shore on the right.  Between two situations, the boat is shown with the farmer and the object that is transported.  The function `rep k s` returns the string `s` repeated `k` times.

In [ ]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    for i in 0 .. n do
        let F = Solution[sprintf "F%d" i]
        let W = Solution[sprintf "W%d" i]
        let G = Solution[sprintf "G%d" i]
        let C = Solution[sprintf "C%d" i]
        printfn "%s%s%s%s%s%s%s%s%s" (rep F "🧑‍🌾") (rep W "🐺") (rep G "🐐") (rep C "🥦") (String.replicate 28 " ")
                                     (rep (1-F) "🧑‍🌾") (rep (1-W) "🐺") (rep (1-G) "🐐") (rep (1-C) "🥦")
        if i < n then
            let sign = if F = 1 then 1 else -1
            let WB = sign * (W - Solution[sprintf "W%d" (i+1)])
            let GB = sign * (G - Solution[sprintf "G%d" (i+1)])
            let CB = sign * (C - Solution[sprintf "C%d" (i+1)])
            printfn "%s🌊  🧑‍🌾%s%s%s 🌊" (String.replicate 12 " ") (rep WB "🐺") (rep GB "🐐") (rep CB "🥦")

In [ ]:
showSolution Solution n